pip install vllm pandas matplotlib

In [ ]:
import os, json, gc
import pandas as pd
import torch
import matplotlib.pyplot as plt

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
import math
from typing import Dict, List, Tuple

In [ ]:
# ====== paths ======
EVAL_MANIFEST = "/root/data/eval_data/manifest.json"

# 要评测的两个模型（建议用 merged bf16，或量化模型路径）
MODEL_SFT = "/root/out/llama-31-8b-pruned-sft-merged-bf16"
MODEL_KD  = "/root/out/llama-31-8b-pruned-kd-merged-bf16"

# vLLM runtime knobs（跑通优先）
DTYPE = "bfloat16"           # 不支持就改 "float16"
GPU_MEMORY_UTIL = 0.90
MAX_MODEL_LEN = 2048

# batch size：越大越快，但显存更吃紧；先用 64/128
BATCH_SIZE = 64

# 评测每个数据集取多少样本（<= 你导出的 jsonl 行数）
N_MAX_PER_DATASET = None     # None 表示用全量（比如导出 2000 就跑 2000）

# logprobs top-k：必须 >= 选项数量（通常 4），给 20 更稳
TOP_LOGPROBS = 20

In [ ]:
def load_manifest(path: str) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def read_jsonl(path: str, n_max=None) -> List[Dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
            if n_max is not None and len(rows) >= n_max:
                break
    return rows

def letters(k: int) -> List[str]:
    return [chr(ord("A")+i) for i in range(k)]

def get_letter_token_ids(tok: AutoTokenizer, k: int) -> Dict[str, List[int]]:
    """
    为了稳：同时考虑 "A" 和 " A" 两种 tokenization。
    返回：每个字母 -> 可能的 token_id 列表（去重）。
    """
    mapping = {}
    for L in letters(k):
        ids = set()
        for s in [L, " " + L]:
            enc = tok.encode(s, add_special_tokens=False)
            # 我们只用单 token 的情况来做 next-token 评分；多 token 直接跳过
            if len(enc) == 1:
                ids.add(enc[0])
        mapping[L] = sorted(list(ids))
    return mapping

def chunked(lst, bs):
    for i in range(0, len(lst), bs):
        yield lst[i:i+bs]

In [ ]:
def eval_mc_next_token_logprob(
    model_path: str,
    manifest_path: str,
    dtype: str = "bfloat16",
    gpu_memory_utilization: float = 0.90,
    max_model_len: int = 2048,
    batch_size: int = 64,
    top_logprobs: int = 20,
    n_max_per_dataset=None,
) -> pd.DataFrame:
    """
    对 manifest 里每个 jsonl 评测 accuracy：
    - prompt 末尾补一个空格
    - 生成 1 token，取该 token 的 logprobs（top-k）
    - 在候选字母 token (A/B/C/...) 中选 logprob 最大的字母
    """
    manifest = load_manifest(manifest_path)

    print(f"Loading tokenizer: {model_path}")
    tok = AutoTokenizer.from_pretrained(model_path, use_fast=True, trust_remote_code=True)

    print(f"Loading vLLM model: {model_path}")
    llm = LLM(
        model=model_path,
        dtype=dtype,
        gpu_memory_utilization=gpu_memory_utilization,
        max_model_len=max_model_len,
        trust_remote_code=True,
    )

    sp = SamplingParams(
        temperature=0.0,
        max_tokens=1,
        logprobs=top_logprobs,
    )

    all_rows = []
    for item in manifest:
        name = item["name"]
        path = item["path"]
        rows = read_jsonl(path, n_max=n_max_per_dataset)
        if len(rows) == 0:
            continue

        # 选项数量可能不同（TruthfulQA 有时 >4）
        k = len(rows[0]["choices"])
        letter_ids = get_letter_token_ids(tok, k)
        cand_letters = letters(k)

        # 构造 prompts
        prompts = []
        answers = []
        for ex in rows:
            prompt = ex["messages"][0]["content"]
            # 关键：末尾加空格，让答案字母更容易成为单 token（" A"）
            if not prompt.endswith(" "):
                prompt = prompt + " "
            prompts.append(prompt)
            answers.append(int(ex["answer_index"]))

        # 推理
        correct = 0
        total = 0
        unknown = 0

        for batch_prompts, batch_answers in zip(chunked(prompts, batch_size), chunked(answers, batch_size)):
            outs = llm.generate(batch_prompts, sp)

            for out, gt in zip(outs, batch_answers):
                total += 1

                # vLLM: out.outputs[0].logprobs is a list (len=max_tokens) of dict[token_id] -> Logprob
                if not out.outputs:
                    unknown += 1
                    continue
                lp_dict = out.outputs[0].logprobs[0]  # first generated token logprobs dict

                # 从候选字母 token id 中找最大 logprob
                best_letter = None
                best_lp = -1e30
                for L in cand_letters:
                    ids = letter_ids.get(L, [])
                    for tid in ids:
                        if tid in lp_dict:
                            lp = lp_dict[tid].logprob
                            if lp > best_lp:
                                best_lp = lp
                                best_letter = L

                # 如果 top-k 里没出现候选 token（偶发），fallback：用生成文本 parse
                if best_letter is None:
                    gen = (out.outputs[0].text or "").strip()
                    if gen:
                        c = gen[0].upper()
                        if c in cand_letters:
                            best_letter = c
                        else:
                            unknown += 1
                            continue
                    else:
                        unknown += 1
                        continue

                pred = ord(best_letter) - ord("A")
                if pred == gt:
                    correct += 1

        acc = correct / max(total - 0, 1)
        all_rows.append({
            "model": os.path.basename(model_path.rstrip("/")),
            "dataset": name,
            "n": total,
            "acc": acc,
            "unknown": unknown,
            "unknown_rate": unknown / max(total, 1),
            "k_choices": k
        })
        print(f"{name:14s}  acc={acc:.4f}  n={total}  unknown={unknown} ({unknown/max(total,1):.2%})")

    df = pd.DataFrame(all_rows)

    # overall (micro-average)
    if len(df) > 0:
        total_n = df["n"].sum()
        overall = (df["acc"] * df["n"]).sum() / max(total_n, 1)
        df_overall = pd.DataFrame([{
            "model": os.path.basename(model_path.rstrip("/")),
            "dataset": "overall_micro",
            "n": int(total_n),
            "acc": float(overall),
            "unknown": int(df["unknown"].sum()),
            "unknown_rate": float(df["unknown"].sum() / max(total_n, 1)),
            "k_choices": None
        }])
        df = pd.concat([df, df_overall], ignore_index=True)

    return df, llm  # 返回 llm 方便显式释放

In [ ]:
df_sft, llm_sft = eval_mc_next_token_logprob(
    model_path=MODEL_SFT,
    manifest_path=EVAL_MANIFEST,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    max_model_len=MAX_MODEL_LEN,
    batch_size=BATCH_SIZE,
    top_logprobs=TOP_LOGPROBS,
    n_max_per_dataset=N_MAX_PER_DATASET,
)

df_sft

In [ ]:
# --- IMPORTANT: release SFT model before evaluating KD ---
del llm_sft
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released SFT model.")

In [ ]:
df_kd, llm_kd = eval_mc_next_token_logprob(
    model_path=MODEL_KD,
    manifest_path=EVAL_MANIFEST,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    max_model_len=MAX_MODEL_LEN,
    batch_size=BATCH_SIZE,
    top_logprobs=TOP_LOGPROBS,
    n_max_per_dataset=N_MAX_PER_DATASET,
)

df_kd

In [ ]:
del llm_kd
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released KD model.")

In [ ]:
df_all = pd.concat([df_sft, df_kd], ignore_index=True)

# 只画 dataset（不含 overall）与 overall 分开画都行
df_plot = df_all.copy()

# 排序：最后一行 overall_micro
order = ["hellaswag", "arc_challenge", "winogrande", "truthfulqa_mc", "overall_micro"]
df_plot["dataset"] = pd.Categorical(df_plot["dataset"], categories=order, ordered=True)
df_plot = df_plot.sort_values(["dataset", "model"])

df_plot

In [ ]:
models = df_plot["model"].unique().tolist()
datasets = [d for d in order if d in df_plot["dataset"].unique().tolist()]

# pivot for plotting
pv = df_plot.pivot(index="dataset", columns="model", values="acc").loc[datasets]

ax = pv.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Accuracy")
ax.set_xlabel("Dataset")
ax.set_title("SFT vs KD (next-token logprob MC accuracy)")
ax.legend(title="Model", loc="best")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()